# Metorial + deepset Haystack Example

This notebook demonstrates how to use Metorial tools with deepset's Haystack framework.

In [ ]:
# Install dependencies
%pip install metorial haystack-ai python-dotenv

In [ ]:
import os

from dotenv import load_dotenv

load_dotenv()

assert os.getenv("METORIAL_API_KEY"), "Set METORIAL_API_KEY"
assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY"
assert os.getenv("EXA_DEPLOYMENT_ID"), "Set EXA_DEPLOYMENT_ID"

In [ ]:
from haystack import Pipeline
from haystack.components.generators.chat import OpenAIChatGenerator
from haystack.components.tools import ToolInvoker
from haystack.dataclasses import ChatMessage

from metorial import Metorial
from metorial.integrations.haystack import create_haystack_tools

In [ ]:
metorial = Metorial(api_key=os.getenv("METORIAL_API_KEY"))

In [ ]:
async def run_pipeline(query: str):
  async with metorial.provider_session(
    provider="openai",
    server_deployments=[os.getenv("EXA_DEPLOYMENT_ID")],
  ) as session:
    tools = create_haystack_tools(session)

    print(f"Available tools: {[t.name for t in tools]}")

    generator = OpenAIChatGenerator(model="gpt-4o", tools=tools)
    tool_invoker = ToolInvoker(tools=tools)

    pipeline = Pipeline()
    pipeline.add_component("generator", generator)
    pipeline.add_component("tool_invoker", tool_invoker)
    pipeline.connect("generator.replies", "tool_invoker.messages")

    messages = [ChatMessage.from_user(query)]
    result = pipeline.run({"generator": {"messages": messages}})

    return result["tool_invoker"]["tool_messages"]

In [ ]:
result = await run_pipeline("Search for the latest Python 3.13 features")
print(result)